# KDHS 2022 — Contraceptive Use Prediction
## Data Cleaning, EDA, Modelling & Deployment
### The Insight Architects Group

---

**Dataset:** Kenya Demographic and Health Survey (KDHS) 2022 — Women's Questionnaire  
**Respondents:** 32,156 women aged 15–49 across all 47 counties  
**Target Variable:** `contraceptive_use` — Type of contraceptive method currently used  
**Methodology:** CRISP-DM  
**Tools:** Python · scikit-learn · LightGBM · SHAP · Flask

---

## 1. Business Understanding

### 1.1 Business Overview

Kenya has made significant progress in family planning over the past three decades, yet contraceptive uptake remains deeply unequal across the country's 47 counties. The 2022 Kenya Demographic and Health Survey (KDHS), a nationally representative dataset of 32,156 women aged 15–49, captures this inequality in unprecedented detail — recording sociodemographic variables such as education, wealth, residence, marital status, and county of residence alongside contraceptive use patterns. Since manual analysis of such a large, multi-variable dataset is impractical for health planners, machine learning can be used to automatically identify the sociodemographic profiles of women most at risk of contraceptive non-use.

This project develops a contraceptive use prediction model using the 2022 KDHS dataset to help the Ministry of Health and family planning partners identify underserved population segments and deploy targeted interventions efficiently and at scale.

### 1.2 Problem Statement

Kenya's modern contraceptive prevalence rate among married women is approximately 60% nationally, but this average conceals major geographic and socioeconomic inequalities. In arid and semi-arid counties such as Turkana, West Pokot, and Mandera, contraceptive uptake falls 20–30 percentage points below the national average, while more urbanised counties like Nairobi and Kiambu report markedly higher rates. Without a data-driven segmentation of at-risk populations, health interventions are distributed uniformly rather than targeted where they are needed most.

**Key challenges include:**

- **Class imbalance** — modern method users (38%) significantly outnumber folkloric and traditional method users (4% combined)
- **High dimensionality** — 5,925 variables in the raw survey requiring careful feature selection down to the most relevant predictors
- **Multicollinearity** — closely related variables such as age, children ever born, and marital status overlap in predictive power
- **Categorical encoding** — ordinal variables like wealth index and education level require careful treatment to preserve their natural ordering

**Objective:** Our objective is to build a model that accurately predicts contraceptive use among Kenyan women aged 15–49 using sociodemographic features, and to identify which factors most strongly drive or prevent modern contraceptive uptake.

### 1.3 Stakeholders

| Stakeholder | How They Use This Model |
|---|---|
| Kenya Ministry of Health — Division of Reproductive Health | Identify counties and demographic groups with highest predicted non-use; allocate field officers and resources accordingly |
| County Health Management Teams (all 47 counties) | Generate county-specific risk profiles to guide local family planning outreach programmes |
| UNFPA Kenya Country Office | Target funding and technical assistance toward highest-risk population segments |
| USAID Kenya | Use model outputs to evaluate programme effectiveness and redirect investments to underserved regions |
| FP2030 Programme | Monitor progress toward family planning targets by tracking predicted non-use rates over time |
| Community Health Promoters (CHPs) | Use county-level risk scores to prioritise household visits in high-risk clusters |

### 1.4 Success Metrics

The model will be considered successful if:

- **Accuracy ≥ 78%** on the held-out test set
- **ROC-AUC ≥ 0.82**, measuring the model's ability to distinguish users from non-users
- **Macro F1-Score ≥ 0.75**, ensuring minority classes are not neglected
- **Balanced Precision and Recall**, especially for non-use prediction — failing to identify at-risk women has real public health consequences
- **SHAP explanations identify at least 5 key features** that policymakers can directly act on
- The model generalises to unseen data via stratified 5-fold cross-validation
- The model improves over a simple logistic regression baseline

### 1.5 Key Business Questions

1. What is the overall distribution of contraceptive use types (modern, traditional, folkloric, none) among Kenyan women aged 15–49?
2. Which counties have the highest predicted rates of contraceptive non-use, and what sociodemographic profiles characterise these women?
3. How do education level and wealth index jointly influence the likelihood of modern contraceptive use?
4. Does urban or rural residence remain a significant predictor of contraceptive use when controlling for wealth and education?
5. Which machine learning model best predicts contraceptive use, and which sociodemographic features are the strongest predictors according to SHAP analysis?

---

## Notebook Structure

| # | Section | CRISP-DM Phase |
|---|---|---|
| 1 | Setup & Imports | — |
| 2 | Data Loading & Initial Inspection | Data Understanding |
| 3 | Pre-Cleaning Audit | Data Understanding |
| 4 | Data Cleaning Pipeline | Data Preparation |
| 5 | Post-Cleaning Validation | Data Preparation |
| 6 | Before vs. After Comparison | Data Preparation |
| 7 | EDA — Univariate | Data Understanding |
| 8 | EDA — Bivariate | Data Understanding |
| 9 | EDA — Multivariate | Data Understanding |
| 10 | Key EDA Findings & Modelling Implications | Data Understanding |
| 11 | Feature Engineering | Data Preparation |
| 12 | Train / Test Split & Class Imbalance Handling | Data Preparation |
| 13 | Baseline Model — Logistic Regression | Modelling |
| 14 | Model 2 — Random Forest | Modelling |
| 15 | Model 3 — LightGBM (Tuned) | Modelling |
| 16 | Model Comparison & Selection | Evaluation |
| 17 | SHAP Explainability | Evaluation |
| 18 | Business Recommendations | Evaluation |
| 19 | Deployment — Flask API | Deployment |

## 2. Setup & Imports

In [2]:
! pip install shap

  Using cached shap-0.44.1-cp38-cp38-win_amd64.whl (450 kB)
  Using cached slicer-0.0.7-py3-none-any.whl (14 kB)
  Using cached numba-0.58.1-cp38-cp38-win_amd64.whl (2.6 MB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)


In [1]:
# Core
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings, joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (9, 5)
os.makedirs('images', exist_ok=True)

# Modelling
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve)
import shap

print('Libraries imported successfully')

Libraries imported successfully


## 3. Data Understanding

The raw file has 5,925 columns. We have loaded only the 17 theory-driven predictors plus the target and the unique respondent id `caseid`. Variable codes were verified against the **DHS-7 Standard Recode Manual**.

#### Data Loading

In [3]:
# Selected DHS variables (verified against the DHS-7 Recode Manual)
cols_needed = ['caseid','v012','v013','v024','v025','v106','v190','v501','v502',
               'v201','v218','v228','v714','v136','v130','v151',
               'v701','v212','v302a','v313']

df = pd.read_csv('KDHS_2022_women.csv', usecols=cols_needed, low_memory=False)

df.rename(columns={
    'v012':'age', 'v013':'age_group', 'v024':'county', 'v025':'residence_type',
    'v106':'education_level', 'v190':'wealth_index', 'v501':'marital_status',
    'v502':'union_status', 'v201':'children_ever_born', 'v218':'living_children',
    'v228':'pregnancy_loss', 'v714':'currently_working', 'v136':'household_size',
    'v130':'religion', 'v151':'household_head_sex', 'v701':'partner_education',
    'v212':'age_first_birth', 'v302a':'ever_used_contraceptive', 'v313':'contraceptive_use'
}, inplace=True)

print('Shape:', df.shape)
df.head()

Shape: (32156, 20)


,caseid,age,age_group,county,residence_type,education_level,religion,household_size,household_head_sex,wealth_index,children_ever_born,age_first_birth,living_children,pregnancy_loss,ever_used_contraceptive,contraceptive_use,marital_status,union_status,partner_education,currently_working
0,1 4 2,34,4,1,1,0,7,6,1,4,4,21.0,4,0,0,0,1,1,0.0,0
1,1 7 2,38,5,1,1,2,1,3,1,5,1,22.0,1,1,2,3,1,1,3.0,0
2,1 10 1,33,4,1,1,1,1,2,2,5,1,21.0,1,0,2,3,4,2,NaN,1
3,1 13 2,39,5,1,1,2,1,8,1,5,5,19.0,5,0,2,3,1,1,1.0,1
4,1 20 2,30,4,1,1,1,96,4,1,4,2,20.0,2,0,0,0,2,1,1.0,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32156 entries, 0 to 32155
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   caseid                   32156 non-null  object 
 1   age                      32156 non-null  int64  
 2   age_group                32156 non-null  int64  
 3   county                   32156 non-null  int64  
 4   residence_type           32156 non-null  int64  
 5   education_level          32156 non-null  int64  
 6   religion                 32156 non-null  int64  
 7   household_size           32156 non-null  int64  
 8   household_head_sex       32156 non-null  int64  
 9   wealth_index             32156 non-null  int64  
 10  children_ever_born       32156 non-null  int64  
 11  age_first_birth          23343 non-null  float64
 12  living_children          32156 non-null  int64  
 13  pregnancy_loss           32156 non-null  int64  
 14  ever_used_contraceptiv

#### Data preview

In [5]:
# 4.1 Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

Missing values per column:
age_first_birth       8813
partner_education    14039
dtype: int64


In [6]:
# 4.2 Target distribution (raw codes: 0 None, 1 Folkloric, 2 Traditional, 3 Modern)
target_labels = {0:'No method', 1:'Folkloric', 2:'Traditional', 3:'Modern'}
print('Target (contraceptive_use) distribution:')
print(df['contraceptive_use'].map(target_labels).value_counts())
print()
print((df['contraceptive_use'].map(target_labels).value_counts(normalize=True)*100).round(1))

Target (contraceptive_use) distribution:
No method      18694
Modern         12195
Traditional     1214
Folkloric         53
Name: contraceptive_use, dtype: int64

No method      58.1
Modern         37.9
Traditional     3.8
Folkloric       0.2
Name: contraceptive_use, dtype: float64


In [ ]:
# 4.3 Are the 274 "duplicates" real duplicates, or just women with similar profiles?

# pandas flags 274 rows that repeat an earlier row (this is what .duplicated() counts):
print('Rows pandas would call duplicates:', df.drop(columns='caseid').duplicated().sum())

# Now marking all rows involved in a look-alike (both the original AND its copy):
look_alike = df.drop(columns='caseid').duplicated(keep=False)
print('Total rows that have a look-alike:', look_alike.sum())

# If these were real duplicates, the IDs would repeat.
# Checking how many DIFFERENT women (unique caseids) these rows actually are:
print('Number of different women among them:', df.loc[look_alike, 'caseid'].nunique())

Rows pandas would call duplicates: 274
Total rows that have a look-alike: 532
Number of different women among them: 532


**Finding: there are no real duplicates.**

pandas flags 274 rows as "duplicates," and in total 532 rows share a look-alike profile. But when we check their IDs, all 532 belong to **different women** (532 unique `caseid`s). They simply share the same age, county, education, and so on.

It is like two students who are both "16, from Nairobi, in Form 3" , they look the same on paper, but they are still two different people with two different admission numbers (`caseid`).

So we have not dropped these rows. Deleting them would throw away real survey responses. No rows are removed.

## 4. Data Cleaning  

For our cleaning we made three deliberate, well-reasoned decisions:
1. **Decoding** the numeric DHS codes into readable labels.
2. **Treating the structural missing data correctly** — the missing values are "not applicable", not gaps.
3. **Keeping the apparent duplicates** (justified above).

#### 4.1 Decoding categorical codes
Codes for the standard variables were confirmed in the DHS-7 manual. `county` and `religion` use Kenya's country-specific codes (constitutional county order 1=Mombasa … 47=Nairobi).

In [8]:
label_maps = {
    'residence_type':     {1:'Urban', 2:'Rural'},
    'education_level':    {0:'No education', 1:'Primary', 2:'Secondary', 3:'Higher'},
    'wealth_index':       {1:'Poorest', 2:'Poorer', 3:'Middle', 4:'Richer', 5:'Richest'},
    'household_head_sex': {1:'Male', 2:'Female'},
    'currently_working':  {0:'No', 1:'Yes'},
    'pregnancy_loss':     {0:'No', 1:'Yes'},
    'marital_status':     {0:'Never married', 1:'Married', 2:'Living together',
                           3:'Widowed', 4:'Divorced', 5:'Separated'},
    'union_status':       {0:'Never in union', 1:'Currently in union', 2:'Formerly in union'},
    'age_group':          {1:'15-19',2:'20-24',3:'25-29',4:'30-34',5:'35-39',6:'40-44',7:'45-49'},
    'contraceptive_use':  {0:'No method', 1:'Folkloric', 2:'Traditional', 3:'Modern'},
    'religion':           {1:'Roman Catholic', 2:'Protestant/Other Christian',
                           3:'Evangelical/Born Again', 4:'African Instituted Church',
                           5:'Orthodox', 7:'Muslim', 8:'Hindu', 9:'Traditionalist',
                           10:'No religion', 96:'Other'},
}
county_map = {1:'Mombasa',2:'Kwale',3:'Kilifi',4:'Tana River',5:'Lamu',6:'Taita Taveta',
 7:'Garissa',8:'Wajir',9:'Mandera',10:'Marsabit',11:'Isiolo',12:'Meru',13:'Tharaka-Nithi',
 14:'Embu',15:'Kitui',16:'Machakos',17:'Makueni',18:'Nyandarua',19:'Nyeri',20:'Kirinyaga',
 21:"Murang'a",22:'Kiambu',23:'Turkana',24:'West Pokot',25:'Samburu',26:'Trans Nzoia',
 27:'Uasin Gishu',28:'Elgeyo Marakwet',29:'Nandi',30:'Baringo',31:'Laikipia',32:'Nakuru',
 33:'Narok',34:'Kajiado',35:'Kericho',36:'Bomet',37:'Kakamega',38:'Vihiga',39:'Bungoma',
 40:'Busia',41:'Siaya',42:'Kisumu',43:'Homa Bay',44:'Migori',45:'Kisii',46:'Nyamira',47:'Nairobi'}

for col, m in label_maps.items():
    df[col] = df[col].map(m)
df['county'] = df['county'].map(county_map)

# Ever-used (V302A): behavioural pattern variable, decoded for EDA only.
df['ever_used_contraceptive'] = df['ever_used_contraceptive'].map(
    {0:'Never used', 1:'Used previously', 2:'Currently using'})

print('Decoded. Example rows:')
df[['county','education_level','wealth_index','marital_status','contraceptive_use']].head()

Decoded. Example rows:


,county,education_level,wealth_index,marital_status,contraceptive_use
0,Mombasa,No education,Richer,Married,No method
1,Mombasa,Secondary,Richest,Married,Modern
2,Mombasa,Primary,Richest,Divorced,Modern
3,Mombasa,Secondary,Richest,Married,Modern
4,Mombasa,Primary,Richer,Living together,No method


#### 4.2 Checking for Missing Values

Two columns have missing values — but the gaps are **not mistakes** in the data. They are simply cases where the question **did not apply** to that woman. So instead of deleting rows or guessing values, we have handled each one in a way that makes sense.

**`age_first_birth` — 8,813 missing.** This is the age a woman was when she had her first child. Every single missing value belongs to a woman who has **never had a child** — so of course she has no "age at first birth." It is blank for a real reason, not by error.
What we did: we created a new yes/no column `has_given_birth` to record whether she has ever given birth, then filled the blank with `0` to mean "not applicable." (Filling it with an average age instead would pretend these women had given birth, which is wrong.)

**`partner_education` — 14,039 missing.** This is the education level of a woman's partner. Most of these blanks (13,844) are women who have **no current partner** — so there is no partner's education to record. Only about 195 are truly missing.
What we did: instead of guessing, we added a clear new category called **"No partner"** so the information is honest and complete.

After this step, the dataset has **zero** missing values.

In [ ]:
# --- age_first_birth ---
# Confirming that every missing value is a woman with no children.
missing_afb = df['age_first_birth'].isna()
print('age_first_birth missing:', missing_afb.sum(),
      '| of those with 0 children:', (df.loc[missing_afb, 'children_ever_born'] == 0).sum())

# Making a yes/no column for "has she ever given birth?", then fill the blank with 0 = "not applicable".
df['has_given_birth'] = df['age_first_birth'].notna().astype(int)
df['age_first_birth'] = df['age_first_birth'].fillna(0)

# --- partner_education ---
# Turning the codes into labels, and fill the blanks with a clear "No partner" category.
df['partner_education'] = df['partner_education'].map(
    {0: 'No education', 1: 'Primary', 2: 'Secondary', 3: 'Higher'}).fillna('No partner')

# Checking: there should now be no missing values left anywhere.
print('Remaining missing values in full frame:', int(df.isnull().sum().sum()))

age_first_birth missing: 8813 | of those with 0 children: 8813
Remaining missing values in full frame: 0


#### 4.3 Keeping all rows (no de-duplication) 

In [10]:
print('Rows retained (no rows dropped):', df.shape[0])

Rows retained (no rows dropped): 32156


#### 4.4 Post-Cleaning Validation

In [11]:
assert df.isnull().sum().sum() == 0, 'There are still missing values!'
assert df.shape[0] == 32156, 'Row count changed unexpectedly!'
print('Validation passed.')
print('Final shape:', df.shape)
print('\nDtypes:\n', df.dtypes)

Validation passed.
Final shape: (32156, 21)

Dtypes:
 caseid                      object
age                          int64
age_group                   object
county                      object
residence_type              object
education_level             object
religion                    object
household_size               int64
household_head_sex          object
wealth_index                object
children_ever_born           int64
age_first_birth            float64
living_children              int64
pregnancy_loss              object
ever_used_contraceptive     object
contraceptive_use           object
marital_status              object
union_status                object
partner_education           object
currently_working           object
has_given_birth              int32
dtype: object


#### 4.5. Before cleaning and after cleaning  Comparison

In [12]:
summary = pd.DataFrame({
    'Stage': ['Raw', 'Cleaned'],
    'Rows': [32156, df.shape[0]],
    'Columns': [20, df.shape[1]],
    'Missing values': [8813+14039, int(df.isnull().sum().sum())],
})
summary

,Stage,Rows,Columns,Missing values
0,Raw,32156,20,22852
1,Cleaned,32156,21,0
